<a href="https://colab.research.google.com/github/mdonbruce/AspNetDocs/blob/master/02_instructor_executed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 2: Secure File Processing with Exceptions — Instructor (Executed)

**Objective:** Validate schema, safely parse amounts, and compute approved USD totals.

**Dataset:** `transaction_records_large.csv`

This notebook is a complete reference solution with outputs.

In [1]:
import pandas as pd, numpy as np

df = pd.read_csv('transaction_records_large.csv')
df.head()

,transaction_id,timestamp,user_id,merchant,amount,currency,method,status,suspect_record
0,TXN-000001,2025-01-02T23:40:01,U5365,Litware Outlet,34.01,CAD,ach,approved,False
1,TXN-000002,2025-06-10T06:40:50,U7432,Contoso Shop,18.04,USD,card,approved,False
2,TXN-000003,2025-06-10T11:12:50,U6117,Contoso Shop,9.58,CAD,wire,approved,False
3,TXN-000004,2025-03-20T06:58:44,U2402,Contoso Shop,45.97,EUR,card,approved,False
4,TXN-000005,2025-03-02T09:27:12,U1941,Adventure Works,19.38,EUR,crypto,approved,False


In [2]:
required_cols = ['transaction_id','timestamp','user_id','merchant','amount','currency','method','status']
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')
print('Schema OK')

Schema OK


In [3]:
def safe_amount(x):
    if x is None:
        return None
    s = str(x).strip()
    if s == '':
        return None
    s = s.replace('$','').replace(',','')
    try:
        v = float(s)
        if not np.isfinite(v):
            return None
        return float(v)
    except Exception:
        return None

df['amount_num'] = df['amount'].apply(safe_amount)
df[['amount','amount_num']].head()

,amount,amount_num
0,34.01,34.01
1,18.04,18.04
2,9.58,9.58
3,45.97,45.97
4,19.38,19.38


In [5]:
approved_usd = df[(df['status']=='approved') & (df['currency']=='USD')]
total = approved_usd['amount_num'].dropna().sum()
bad_count = int(approved_usd['amount_num'].isna().sum())
print('Total approved USD volume:', round(total,2))
print('Invalid amount rows (approved USD):', bad_count)

Total approved USD volume: 203639.71
Invalid amount rows (approved USD): 140
